# 06 — AML Graph Analysis & Results

This notebook computes:
1. Anomaly scores for all node embeddings (via PyTorch GAN)
2. Graph statistics (degree distribution, connectivity)
3. Transaction value / money flow analysis
4. Suspicious node identification with financial impact

**Outputs saved to parquet** for downstream use by the dashboard and pattern analysis.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import networkx as nx
import torch

from gan_anomaly import Generator, Encoder, anomaly_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Pipeline integration ──────────────────────────────────────────
_RUN_DIR = os.environ.get("AML_RUN_DIR", "")
DATA_DIR = os.path.join(_RUN_DIR, "data") if _RUN_DIR else "data"
EMBEDDINGS_DIR = os.path.join(_RUN_DIR, "embeddings") if _RUN_DIR else "embeddings"
MODELS_DIR = os.path.join(_RUN_DIR, "models") if _RUN_DIR else "models"
RESULTS_DIR = os.path.join(_RUN_DIR, "results") if _RUN_DIR else "results"

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Device: {device}")

## 1. Load Data

In [ ]:
# Load processed data from notebook 01
transactions = pd.read_parquet(os.path.join(DATA_DIR, "transactions_processed.parquet"))
node_features = pd.read_parquet(os.path.join(DATA_DIR, "node_features.parquet"))
edges = pd.read_parquet(os.path.join(DATA_DIR, "edges.parquet"))

# Load embeddings from notebook 02
embeddings = np.load(os.path.join(EMBEDDINGS_DIR, "node_embeddings.npy"))
node_ids = np.load(os.path.join(EMBEDDINGS_DIR, "node_ids.npy"), allow_pickle=True)

# Load GAN models from notebook 03
with open(os.path.join(MODELS_DIR, "training_meta.json")) as f:
    meta = json.load(f)

G_model = Generator(meta["latent_dim"], meta["input_dim"], meta["g_hidden"], meta["n_layers"], meta["activation"]).to(device)
E_model = Encoder(meta["input_dim"], meta["latent_dim"], meta["e_hidden"], meta["n_layers"], meta["activation"]).to(device)
G_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, "generator.pt"), map_location=device, weights_only=True))
E_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, "encoder.pt"), map_location=device, weights_only=True))
G_model.eval(); E_model.eval()

# Load threshold from training
X_train = np.load(os.path.join(MODELS_DIR, "X_train.npy"))
train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
train_scores = anomaly_score(train_tensor, E_model, G_model).cpu().numpy()
threshold = np.percentile(train_scores, 99)

print(f"Loaded {len(transactions):,} transactions, {len(node_features):,} nodes")
print(f"Embeddings: {embeddings.shape}")
print(f"Anomaly threshold (P99): {threshold:.6f}")

In [ ]:
# Compute anomaly scores for all nodes
all_tensor = torch.tensor(embeddings, dtype=torch.float32).to(device)
anomaly_scores = anomaly_score(all_tensor, E_model, G_model).cpu().numpy()

# ── Free GPU immediately — rest of notebook is CPU only ──
del all_tensor, train_tensor, G_model, E_model
torch.cuda.empty_cache()
import gc; gc.collect()

# Build enriched node dataframe
node_emb_df = pd.DataFrame({"id": node_ids})
for i in range(embeddings.shape[1]):
    node_emb_df[f"emb_{i}"] = embeddings[:, i]
node_emb_df["anomaly_score"] = anomaly_scores
node_emb_df["is_anomaly"] = anomaly_scores > threshold

score_min, score_max = anomaly_scores.min(), anomaly_scores.max()
if score_max > score_min:
    node_emb_df["risk_score"] = (anomaly_scores - score_min) / (score_max - score_min)
else:
    node_emb_df["risk_score"] = 0.0

# Merge SAR labels
node_emb_df = node_emb_df.merge(node_features[["id", "is_sar"]], on="id", how="left")
node_emb_df["is_sar"] = node_emb_df["is_sar"].fillna(0).astype(int)

# Save enriched embeddings
node_emb_df.to_parquet(os.path.join(RESULTS_DIR, "node_embeddings_scored.parquet"), index=False)

print(f"Anomaly scores computed! GPU freed.")
print(f"Score range: [{anomaly_scores.min():.6f}, {anomaly_scores.max():.6f}]")
print(f"Threshold: {threshold:.6f}")
print(f"Anomalies: {node_emb_df['is_anomaly'].sum()} / {len(node_emb_df)}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e6:.1f} MB")

In [5]:
# Create NetworkX graph from full transactions
G = nx.from_pandas_edgelist(
    transactions,
    source="source",
    target="target",
    create_using=nx.DiGraph()
)

print(f"Graph Statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.4f}")
print(f"  Is connected: {nx.is_weakly_connected(G)}")
print(f"  Connected components: {nx.number_weakly_connected_components(G)}")

Graph Statistics:
  Nodes: 7477
  Edges: 67100
  Density: 0.0012
  Is connected: True
  Connected components: 1


## 2. Transaction Value Analysis

In [6]:
print("Transaction Amount Statistics:")
print("=" * 50)
print(f"Total transactions: {len(transactions):,}")
print(f"Total value: ${transactions['base_amt'].sum():,.2f}")
print(f"Average: ${transactions['base_amt'].mean():,.2f}")
print(f"Median: ${transactions['base_amt'].median():,.2f}")
print(f"Min: ${transactions['base_amt'].min():,.2f}")
print(f"Max: ${transactions['base_amt'].max():,.2f}")
print("=" * 50)

Transaction Amount Statistics:
Total transactions: 430,744
Total value: $231,281,010.69
Average: $536.93
Median: $493.24
Min: $79.97
Max: $9,489.83


In [7]:
# Calculate money flow per node
outgoing_amounts = transactions.groupby("source")["base_amt"].sum().rename("outgoing_amt")
incoming_amounts = transactions.groupby("target")["base_amt"].sum().rename("incoming_amt")
outgoing_counts = transactions.groupby("source").size().rename("outgoing_count")
incoming_counts = transactions.groupby("target").size().rename("incoming_count")

node_money = node_emb_df[["id", "anomaly_score", "is_anomaly", "is_sar"]].copy()
node_money = node_money.merge(outgoing_amounts, left_on="id", right_index=True, how="left")
node_money = node_money.merge(incoming_amounts, left_on="id", right_index=True, how="left")
node_money = node_money.merge(outgoing_counts, left_on="id", right_index=True, how="left")
node_money = node_money.merge(incoming_counts, left_on="id", right_index=True, how="left")
node_money = node_money.fillna(0)

node_money["total_volume"] = node_money["outgoing_amt"] + node_money["incoming_amt"]
node_money["net_flow"] = node_money["incoming_amt"] - node_money["outgoing_amt"]
node_money["total_transactions"] = node_money["outgoing_count"] + node_money["incoming_count"]

print("Money Flow Statistics:")
print("=" * 50)
print(f"Total money volume: ${node_money['total_volume'].sum()/2:,.2f}")
print(f"Avg volume per node: ${node_money['total_volume'].mean():,.2f}")
print(f"Max volume (single node): ${node_money['total_volume'].max():,.2f}")
print("=" * 50)

Money Flow Statistics:
Total money volume: $231,281,010.69
Avg volume per node: $61,674.94
Max volume (single node): $123,636,868.40


In [8]:
# Compare money flow: Anomalous vs Normal nodes
anomalous_money = node_money[node_money["is_anomaly"] == True]
normal_money = node_money[node_money["is_anomaly"] == False]

print("Money Flow Comparison: Anomalous vs Normal Nodes")
print("=" * 60)
print(f"{'Metric':<30} {'Anomalous':>15} {'Normal':>15}")
print("-" * 60)
print(f"{'Node Count':<30} {len(anomalous_money):>15,} {len(normal_money):>15,}")
print(f"{'Total Volume ($)':<30} {anomalous_money['total_volume'].sum():>15,.0f} {normal_money['total_volume'].sum():>15,.0f}")
print(f"{'Avg Volume per Node ($)':<30} {anomalous_money['total_volume'].mean():>15,.0f} {normal_money['total_volume'].mean():>15,.0f}")
print(f"{'Avg Transactions per Node':<30} {anomalous_money['total_transactions'].mean():>15,.1f} {normal_money['total_transactions'].mean():>15,.1f}")
print(f"{'Avg Outgoing ($)':<30} {anomalous_money['outgoing_amt'].mean():>15,.0f} {normal_money['outgoing_amt'].mean():>15,.0f}")
print(f"{'Avg Incoming ($)':<30} {anomalous_money['incoming_amt'].mean():>15,.0f} {normal_money['incoming_amt'].mean():>15,.0f}")
print("=" * 60)

Money Flow Comparison: Anomalous vs Normal Nodes
Metric                               Anomalous          Normal
------------------------------------------------------------
Node Count                                  89           7,411
Total Volume ($)                   360,162,029     102,399,992
Avg Volume per Node ($)              4,046,764          13,817
Avg Transactions per Node              7,554.6            25.5
Avg Outgoing ($)                     2,235,009           4,367
Avg Incoming ($)                     1,811,755           9,450


In [9]:
# Top 20 suspicious nodes by transaction volume
top_suspicious = node_money[node_money["is_anomaly"] == True].nlargest(20, "total_volume")

print("Top 20 Suspicious Nodes by Transaction Volume:")
print("=" * 80)
cols = ["id", "is_sar", "total_volume", "outgoing_amt", "incoming_amt", "total_transactions", "anomaly_score"]
print(top_suspicious[cols].to_string(index=False))

total_suspicious = anomalous_money["total_volume"].sum() / 2
print(f"\nTotal money flowing through suspicious nodes: ${total_suspicious:,.2f}")

Top 20 Suspicious Nodes by Transaction Volume:
      id  is_sar  total_volume  outgoing_amt  incoming_amt  total_transactions  anomaly_score
c5dccde2       0  123636868.40   87578767.05   36058101.35            231672.0     761.507080
34cdefef       0   73719585.36   31185531.35   42534054.01            138169.0     515.544678
c1bfb464       1   33377783.08   17051866.28   16325916.80             62545.0     306.655273
c613c146       0   21217613.62   10993727.32   10223886.30             39628.0     236.608902
7b808486       0   15354830.15    8014586.82    7340243.33             28769.0     170.377472
308e3438       0   11772258.44    5902121.83    5870136.61             22039.0     142.306152
44142e26       0    9441174.30    4764473.67    4676700.63             17647.0     100.935349
8712392d       0    7858308.48    3903399.24    3954909.24             14668.0      95.306870
7af63fa1       0    6592872.78    3238434.44    3354438.34             12308.0      67.582428
390e2f47     

## 3. Degree Analysis

In [10]:
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())
total_degrees = {n: in_degrees.get(n, 0) + out_degrees.get(n, 0) for n in G.nodes()}

print(f"Degree Statistics:")
print(f"  In-degree  — mean: {np.mean(list(in_degrees.values())):.2f}, max: {max(in_degrees.values())}")
print(f"  Out-degree — mean: {np.mean(list(out_degrees.values())):.2f}, max: {max(out_degrees.values())}")
print(f"  Total      — mean: {np.mean(list(total_degrees.values())):.2f}, max: {max(total_degrees.values())}")

print(f"\nTop 10 nodes by total degree:")
top_degree = sorted(total_degrees.items(), key=lambda x: x[1], reverse=True)[:10]
for node, degree in top_degree:
    print(f"  {node}: {degree}")

Degree Statistics:
  In-degree  — mean: 8.97, max: 2884
  Out-degree — mean: 8.97, max: 6742
  Total      — mean: 17.95, max: 9626

Top 10 nodes by total degree:
  c5dccde2: 9626
  34cdefef: 6245
  c1bfb464: 4436
  c613c146: 3478
  7b808486: 2799
  308e3438: 2377
  44142e26: 1986
  8712392d: 1775
  7af63fa1: 1561
  390e2f47: 1423


## 4. Anomaly Score Analysis

In [11]:
print("Anomaly Score Distribution:")
print(f"  Mean:   {node_emb_df['anomaly_score'].mean():.6f}")
print(f"  Median: {node_emb_df['anomaly_score'].median():.6f}")
print(f"  Std:    {node_emb_df['anomaly_score'].std():.6f}")
print(f"  Min:    {node_emb_df['anomaly_score'].min():.6f}")
print(f"  Max:    {node_emb_df['anomaly_score'].max():.6f}")
print(f"  Threshold: {threshold:.6f}")

sar_scores = node_emb_df[node_emb_df["is_sar"] == 1]["anomaly_score"]
non_sar_scores = node_emb_df[node_emb_df["is_sar"] == 0]["anomaly_score"]
print(f"\n  SAR nodes — mean: {sar_scores.mean():.6f}, median: {sar_scores.median():.6f}")
print(f"  Non-SAR   — mean: {non_sar_scores.mean():.6f}, median: {non_sar_scores.median():.6f}")

Anomaly Score Distribution:
  Mean:   1.072875
  Median: 0.385579
  Std:    12.161880
  Min:    0.067236
  Max:    761.507080
  Threshold: 5.807745

  SAR nodes — mean: 1.849449, median: 0.589691
  Non-SAR   — mean: 0.999930, median: 0.372972


In [12]:
# Summary
total_nodes = len(node_emb_df)
anomaly_count = node_emb_df["is_anomaly"].sum()

print("=" * 50)
print("ANOMALY DETECTION SUMMARY")
print("=" * 50)
print(f"Total nodes analyzed: {total_nodes}")
print(f"Anomalies detected:   {anomaly_count} ({100*anomaly_count/total_nodes:.1f}%)")
print(f"Normal nodes:         {total_nodes - anomaly_count} ({100*(total_nodes-anomaly_count)/total_nodes:.1f}%)")
print(f"\nAnomaly threshold:    {threshold:.6f}")
print(f"Mean anomaly score:   {node_emb_df['anomaly_score'].mean():.6f}")
print("=" * 50)

sar_nodes = node_emb_df["is_sar"].sum()
detected_sar = ((node_emb_df["is_sar"] == 1) & (node_emb_df["is_anomaly"])).sum()
print(f"\nSAR Labels in data:       {sar_nodes}")
print(f"SAR detected as anomaly:  {detected_sar} ({100*detected_sar/max(sar_nodes,1):.1f}%)")

ANOMALY DETECTION SUMMARY
Total nodes analyzed: 7500
Anomalies detected:   89 (1.2%)
Normal nodes:         7411 (98.8%)

Anomaly threshold:    5.807745
Mean anomaly score:   1.072875

SAR Labels in data:       644
SAR detected as anomaly:  18 (2.8%)


## 5. Top Anomalous Nodes

In [13]:
top_anomalies = node_emb_df.nlargest(20, "anomaly_score")[["id", "is_sar", "anomaly_score", "is_anomaly"]]
print("Top 20 Most Anomalous Nodes:")
print("=" * 60)
print(top_anomalies.to_string(index=False))

Top 20 Most Anomalous Nodes:
      id  is_sar  anomaly_score  is_anomaly
c5dccde2       0     761.507080        True
34cdefef       0     515.544678        True
c1bfb464       1     306.655273        True
c613c146       0     236.608902        True
7b808486       0     170.377472        True
308e3438       0     142.306152        True
0ed1750d       1     124.536407        True
44142e26       0     100.935349        True
8712392d       0      95.306870        True
7af63fa1       0      67.582428        True
390e2f47       0      63.043407        True
4de191f1       0      44.377811        True
70d9830f       0      42.527370        True
b59bd631       1      42.441689        True
3499edce       1      39.349297        True
ab35a7fd       1      35.824970        True
e6ccbd74       1      34.444618        True
b5f390f8       0      33.794964        True
3bdc1134       0      33.525276        True
c09d1681       1      32.772373        True


In [14]:
# Top anomalous node neighborhood
top_node = top_anomalies.iloc[0]["id"]

if top_node in G.nodes():
    neighbors_1 = set(G.predecessors(top_node)) | set(G.successors(top_node))
    neighbors_2 = set()
    for n in neighbors_1:
        neighbors_2 |= set(G.predecessors(n)) | set(G.successors(n))
    
    subgraph_nodes = {top_node} | neighbors_1 | neighbors_2
    G_sub = G.subgraph(subgraph_nodes).copy()
    
    print(f"Top anomaly '{top_node}' neighborhood:")
    print(f"  1-hop neighbors: {len(neighbors_1)}")
    print(f"  2-hop neighbors: {len(neighbors_2 - neighbors_1 - {top_node})}")
    print(f"  Subgraph: {G_sub.number_of_nodes()} nodes, {G_sub.number_of_edges()} edges")
else:
    print(f"Node {top_node} not found in graph")

Top anomaly 'c5dccde2' neighborhood:
  1-hop neighbors: 6918
  2-hop neighbors: 558
  Subgraph: 7477 nodes, 67100 edges


## 6. Summary Report

In [15]:
print("\n" + "=" * 70)
print(" AML ANOMALY DETECTION - ANALYSIS REPORT ")
print("=" * 70)

print(f"\n{'Graph Statistics':^40}")
print("-" * 40)
print(f"  Total Nodes:          {G.number_of_nodes():,}")
print(f"  Total Edges:          {G.number_of_edges():,}")
print(f"  Graph Density:        {nx.density(G):.6f}")
print(f"  Avg Degree:           {sum(total_degrees.values())/len(total_degrees):.2f}")

print(f"\n{'Transaction Value Statistics':^40}")
print("-" * 40)
print(f"  Total Transactions:   {len(transactions):,}")
print(f"  Total Value:          ${transactions['base_amt'].sum():,.2f}")
print(f"  Average Transaction:  ${transactions['base_amt'].mean():,.2f}")
print(f"  Max Transaction:      ${transactions['base_amt'].max():,.2f}")

print(f"\n{'Anomaly Detection Results':^40}")
print("-" * 40)
print(f"  Nodes Analyzed:       {len(node_emb_df):,}")
print(f"  Anomalies Detected:   {node_emb_df['is_anomaly'].sum():,} ({100*node_emb_df['is_anomaly'].mean():.1f}%)")
print(f"  Detection Threshold:  {threshold:.6f}")
print(f"  Max Anomaly Score:    {node_emb_df['anomaly_score'].max():.6f}")

print(f"\n{'Suspicious Money Flow':^40}")
print("-" * 40)
suspicious_volume = node_money[node_money['is_anomaly']]['total_volume'].sum() / 2
total_volume = transactions['base_amt'].sum()
print(f"  Suspicious Volume:    ${suspicious_volume:,.2f}")
print(f"  % of Total Volume:    {100*suspicious_volume/total_volume:.1f}%")

print(f"\n{'Outputs Saved':^40}")
print("-" * 40)
print(f"  - results/node_embeddings_scored.parquet")

print("\n" + "=" * 70)
print(" ANALYSIS COMPLETE ")
print("=" * 70)


 AML ANOMALY DETECTION - ANALYSIS REPORT 

            Graph Statistics            
----------------------------------------
  Total Nodes:          7,477
  Total Edges:          67,100
  Graph Density:        0.001200
  Avg Degree:           17.95

      Transaction Value Statistics      
----------------------------------------
  Total Transactions:   430,744
  Total Value:          $231,281,010.69
  Average Transaction:  $536.93
  Max Transaction:      $9,489.83

       Anomaly Detection Results        
----------------------------------------
  Nodes Analyzed:       7,500
  Anomalies Detected:   89 (1.2%)
  Detection Threshold:  5.807745
  Max Anomaly Score:    761.507080

         Suspicious Money Flow          
----------------------------------------
  Suspicious Volume:    $180,081,014.44
  % of Total Volume:    77.9%

             Outputs Saved              
----------------------------------------
  - results/node_embeddings_scored.parquet

 ANALYSIS COMPLETE 


In [16]:
# ── GPU Cleanup — free VRAM for next notebook ──
import gc
for v in ["G_model", "E_model", "all_tensor", "train_tensor"]:
    if v in dir():
        exec(f"del {v}")
torch.cuda.empty_cache(); gc.collect()
print(f"GPU freed: {torch.cuda.memory_allocated()/1e6:.1f} MB allocated")

GPU freed: 8.5 MB allocated
